# AEGIS — Data Cleaning, Preprocessing & Exploratory Data Analysis

**Project:** AEGIS: Adversarial Enforcement and Guardrail Interception System  
**Course Presentation Notebook**

---

## Overview

AEGIS is a three-layer inline security proxy for healthcare AI agents. Its Layer 1 component uses a fine-tuned DistilBERT classifier (exported to ONNX) to classify incoming prompts into five threat categories before they ever reach the underlying medical LLM.

**Five Classes:**

| ID | Label | Description |
|----|-------|-------------|
| 0 | `benign` | Legitimate clinical queries |
| 1 | `direct_injection` | Prompt injection attacks targeting the system directly |
| 2 | `indirect_injection` | Malicious instructions hidden inside documents / tool output |
| 3 | `jailbreak` | Attempts to override safety constraints or role-play bypass |
| 4 | `phi_extraction` | Attempts to extract Protected Health Information (18 HIPAA identifiers) |

This notebook covers:
1. **Data generation** using the AEGIS preprocessing pipeline  
2. **Data cleaning** — deduplication, length filtering, label validation  
3. **Exploratory Data Analysis (EDA)** — class distribution, text-length distributions, vocabulary, and sample reviews  
4. **Train / Val / Test split** with stratification  
5. **Key takeaways** for the classifier training run

## 1. Setup & Imports

In [ ]:
import json
import random
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

# ── Notebook display settings ───────────────────────────────────────────────
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)
plt.rcParams["figure.dpi"] = 120

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_NAMES = {
    0: "benign",
    1: "direct_injection",
    2: "indirect_injection",
    3: "jailbreak",
    4: "phi_extraction",
}
CLASS_COLORS = {
    "benign": "#4CAF50",
    "direct_injection": "#F44336",
    "indirect_injection": "#FF9800",
    "jailbreak": "#9C27B0",
    "phi_extraction": "#2196F3",
}

print("Python", sys.version)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2. Data Generation

The AEGIS dataset is assembled from multiple public sources plus two synthetic generators. Here we either load pre-built JSONL files (if they exist) or reconstruct the **synthetic** portions directly — no internet access required.

Sources used in the full pipeline:
- `deepset/prompt-injections` (HuggingFace)
- `jackhhao/jailbreak-classification` (HuggingFace)
- `rubend18/ChatGPT-Jailbreak-Prompts` (HuggingFace)
- `GBaker/MedQA-USMLE-4-options` (HuggingFace)
- PubMedQA (GitHub)
- Tensor Trust — hijacking + extraction benchmarks (GitHub)
- HackAPrompt — successful attack submissions (HuggingFace, gated)
- **Synthetic indirect injection** (generated in `scripts/data/preprocess.py`)
- **Synthetic PHI extraction** (generated in `scripts/data/preprocess.py`)

In [ ]:
# Add repo root to path so we can import the preprocessing helpers
REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "scripts" / "data"))

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
ALL_JSONL = PROCESSED_DIR / "all.jsonl"

records: list[dict] = []

if ALL_JSONL.exists():
    print(f"Loading pre-built dataset from {ALL_JSONL}")
    with open(ALL_JSONL) as fh:
        for line in fh:
            records.append(json.loads(line))
    print(f"Loaded {len(records):,} records.")
else:
    print("`data/processed/all.jsonl` not found — generating synthetic portions only.")
    print("Run `python scripts/data/preprocess.py` to build the full dataset.\n")

    from preprocess import generate_indirect_injection_samples, generate_phi_extraction_samples

    records.extend(generate_indirect_injection_samples(seed=RANDOM_SEED))
    records.extend(generate_phi_extraction_samples(seed=RANDOM_SEED))
    print(f"Generated {len(records):,} synthetic records (indirect_injection + phi_extraction).")
    print("NOTE: Public-source classes (benign, direct_injection, jailbreak) require internet access.")

## 3. Build the Raw DataFrame

In [ ]:
df_raw = pd.DataFrame(records)
df_raw["class_name"] = df_raw["label"].map(CLASS_NAMES)
df_raw["text_len"] = df_raw["text"].str.len()
df_raw["word_count"] = df_raw["text"].str.split().str.len()

print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

## 4. Data Cleaning

### 4.1 — Schema validation

In [ ]:
print("=== Missing values ===")
print(df_raw[["text", "label", "source"]].isnull().sum())
print()

print("=== Unexpected label values ===")
valid_labels = set(CLASS_NAMES.keys())
bad_labels = df_raw[~df_raw["label"].isin(valid_labels)]
print(f"{len(bad_labels)} rows with invalid labels")
print()

print("=== Empty or whitespace-only texts ===")
empty_texts = df_raw[df_raw["text"].str.strip().eq("")]
print(f"{len(empty_texts)} rows with empty text")

### 4.2 — Deduplication

In [ ]:
before = len(df_raw)
df_clean = df_raw.drop_duplicates(subset="text").copy()
after = len(df_clean)
removed = before - after

print(f"Rows before dedup : {before:,}")
print(f"Rows after  dedup : {after:,}")
print(f"Duplicates removed: {removed:,} ({removed / before * 100:.2f}%)")

### 4.3 — Minimum length filter

Very short strings (< 10 characters) are unlikely to carry useful signal and may be noise.

In [ ]:
MIN_CHARS = 10
too_short = df_clean[df_clean["text_len"] < MIN_CHARS]
print(f"Rows with < {MIN_CHARS} characters: {len(too_short)}")
if len(too_short):
    print(too_short[["text", "class_name", "source"]].to_string())

df_clean = df_clean[df_clean["text_len"] >= MIN_CHARS].copy()
print(f"\nRecords after length filter: {len(df_clean):,}")

### 4.4 — Text normalisation

Minimal normalisation: strip leading/trailing whitespace and collapse multiple newlines.  
We intentionally keep original casing, punctuation, and special characters — they carry signal for injection detection.

In [ ]:
import re

def normalize_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\n{3,}", "\n\n", text)  # collapse 3+ newlines to 2
    return text

df_clean["text"] = df_clean["text"].apply(normalize_text)
df_clean["text_len"] = df_clean["text"].str.len()
df_clean["word_count"] = df_clean["text"].str.split().str.len()

print("Normalisation complete.")
print(df_clean[["text_len", "word_count"]].describe().round(1))

### 4.5 — Cleaning summary

In [ ]:
print(f"Raw records      : {len(df_raw):>7,}")
print(f"After dedup      : {after:>7,}")
print(f"After len filter : {len(df_clean):>7,}")
print(f"\nRetained         : {len(df_clean) / len(df_raw) * 100:.1f}%")

## 5. Exploratory Data Analysis

### 5.1 — Class distribution

In [ ]:
class_counts = df_clean["class_name"].value_counts().reindex(CLASS_NAMES.values())
class_pct = (class_counts / class_counts.sum() * 100).round(1)

dist_df = pd.DataFrame({"count": class_counts, "pct": class_pct})
print(dist_df.to_string())
print(f"\nTotal: {class_counts.sum():,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = [CLASS_COLORS[c] for c in class_counts.index]

# Bar chart
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor="white", linewidth=0.8)
axes[0].set_title("Class Distribution — Sample Counts", fontweight="bold")
axes[0].set_ylabel("Number of samples")
axes[0].set_xlabel("Class")
axes[0].tick_params(axis="x", rotation=30)
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                 f"{count:,}", ha="center", va="bottom", fontsize=9)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    colors=colors,
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title("Class Distribution — Proportion", fontweight="bold")

plt.tight_layout()
plt.savefig("class_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: class_distribution.png")

**Observation:** The dataset is highly imbalanced. `benign` (~62%) and `direct_injection` (~30%) dominate; `indirect_injection` and `phi_extraction` together account for only ~3.7%. This mirrors real-world threat distributions — most traffic is legitimate. The classifier must maintain low false-positive rates on the benign majority while still detecting the rare but critical attack classes.

### 5.2 — Data sources per class

In [ ]:
source_breakdown = (
    df_clean.groupby(["class_name", "source"])
    .size()
    .reset_index(name="count")
    .sort_values(["class_name", "count"], ascending=[True, False])
)
print(source_breakdown.to_string(index=False))

### 5.3 — Text length distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for class_name, color in CLASS_COLORS.items():
    subset = df_clean[df_clean["class_name"] == class_name]
    if len(subset) == 0:
        continue
    axes[0].hist(subset["text_len"], bins=50, alpha=0.5, color=color, label=class_name, density=True)
    axes[1].hist(subset["word_count"], bins=50, alpha=0.5, color=color, label=class_name, density=True)

axes[0].set_title("Character Length Distribution by Class", fontweight="bold")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Density")
axes[0].axvline(128 * 5, color="red", linestyle="--", linewidth=1.2, label="~128-token boundary")
axes[0].legend(fontsize=8)

axes[1].set_title("Word Count Distribution by Class", fontweight="bold")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Density")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("text_length_distributions.png", bbox_inches="tight")
plt.show()
print("Saved: text_length_distributions.png")

In [ ]:
# Per-class length statistics
length_stats = (
    df_clean.groupby("class_name")[["text_len", "word_count"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(1)
)
length_stats

**Observation:** `indirect_injection` samples are typically longer (they embed a malicious instruction inside a realistic clinical document template). `phi_extraction` prompts tend to be shorter and more direct. The 128-token `max_length` cap in the classifier will truncate only a small fraction of samples.

### 5.4 — Word count box plots

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

order = list(CLASS_NAMES.values())
palette = {k: v for k, v in CLASS_COLORS.items()}

available_classes = df_clean["class_name"].unique()
plot_order = [c for c in order if c in available_classes]

sns.boxplot(
    data=df_clean,
    x="class_name",
    y="word_count",
    order=plot_order,
    palette=palette,
    showfliers=False,
    ax=ax,
)
ax.set_title("Word Count per Class (outliers hidden)", fontweight="bold")
ax.set_xlabel("Class")
ax.set_ylabel("Word count")
ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig("word_count_boxplot.png", bbox_inches="tight")
plt.show()
print("Saved: word_count_boxplot.png")

### 5.5 — Sample examples per class

In [ ]:
rng = random.Random(RANDOM_SEED)

for class_id, class_name in CLASS_NAMES.items():
    subset = df_clean[df_clean["label"] == class_id]
    if len(subset) == 0:
        print(f"\n--- {class_name.upper()} (class {class_id}) --- [no samples in current dataset]")
        continue
    samples = subset.sample(min(3, len(subset)), random_state=RANDOM_SEED)
    print(f"\n{'─' * 70}")
    print(f"  CLASS {class_id}: {class_name.upper()}")
    print(f"{'─' * 70}")
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        preview = row["text"][:300].replace("\n", " ")
        print(f"  [{i}] (source={row['source']}, len={row['text_len']})")
        print(f"      {preview}..." if len(row["text"]) > 300 else f"      {preview}")
        print()

### 5.6 — Most-common keywords per class (top unigrams)

In [ ]:
import re
from collections import Counter

STOPWORDS = {
    "the", "a", "an", "and", "or", "is", "in", "of", "to", "for", "with",
    "this", "that", "it", "be", "are", "was", "were", "on", "at", "by",
    "from", "as", "have", "has", "all", "i", "you", "me", "my", "your",
    "do", "not", "can", "what", "which", "its", "their", "any", "will",
    "please", "about", "up", "out", "if", "into", "so", "no", "more",
}


def top_words(texts: pd.Series, n: int = 15) -> list[tuple[str, int]]:
    tokens = []
    for t in texts:
        for w in re.findall(r"[a-z]+", t.lower()):
            if w not in STOPWORDS and len(w) > 2:
                tokens.append(w)
    return Counter(tokens).most_common(n)


fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(22, 4), sharey=False)

for ax, (class_id, class_name) in zip(axes, CLASS_NAMES.items()):
    subset = df_clean[df_clean["label"] == class_id]
    if len(subset) == 0:
        ax.set_title(f"{class_name}\n(no data)", fontsize=8)
        ax.axis("off")
        continue

    words, counts = zip(*top_words(subset["text"])) if len(subset) else ([], [])
    color = CLASS_COLORS[class_name]

    ax.barh(list(reversed(words)), list(reversed(counts)), color=color, alpha=0.85)
    ax.set_title(class_name.replace("_", "\n"), fontsize=9, fontweight="bold", color=color)
    ax.tick_params(axis="y", labelsize=7)
    ax.tick_params(axis="x", labelsize=7)

plt.suptitle("Top 15 Words per Class (stopwords removed)", fontweight="bold", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("top_words_per_class.png", bbox_inches="tight")
plt.show()
print("Saved: top_words_per_class.png")

### 5.7 — Token truncation analysis

The classifier uses `max_length=128` tokens. Tokens ≈ characters / 4.5 for English text. We estimate the fraction of samples that will be truncated.

In [ ]:
# Rough estimate: avg chars per token for BERT tokeniser ≈ 4.5
CHARS_PER_TOKEN = 4.5
MAX_TOKENS = 128

df_clean["est_tokens"] = (df_clean["text_len"] / CHARS_PER_TOKEN).round().astype(int)
truncated = df_clean[df_clean["est_tokens"] > MAX_TOKENS]

print(f"Estimated truncated samples (>{MAX_TOKENS} tokens): {len(truncated):,} / {len(df_clean):,} = {len(truncated)/len(df_clean)*100:.1f}%")
print()

trunc_by_class = (
    df_clean.assign(truncated=df_clean["est_tokens"] > MAX_TOKENS)
    .groupby("class_name")["truncated"]
    .agg(["sum", "count"])
    .assign(pct=lambda x: (x["sum"] / x["count"] * 100).round(1))
    .rename(columns={"sum": "truncated", "count": "total"})
)
print(trunc_by_class)

## 6. Train / Validation / Test Split

Stratified 80 / 10 / 10 split — each class retains its proportion across splits.

In [ ]:
from collections import defaultdict


def stratified_split(
    df: pd.DataFrame,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = random.Random(seed)
    train_rows, val_rows, test_rows = [], [], []

    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].to_dict("records")
        rng.shuffle(rows)
        n = len(rows)
        t_end = int(n * train_ratio)
        v_end = int(n * (train_ratio + val_ratio))
        train_rows.extend(rows[:t_end])
        val_rows.extend(rows[t_end:v_end])
        test_rows.extend(rows[v_end:])

    return (
        pd.DataFrame(train_rows),
        pd.DataFrame(val_rows),
        pd.DataFrame(test_rows),
    )


df_train, df_val, df_test = stratified_split(df_clean, seed=RANDOM_SEED)

print(f"Train : {len(df_train):>7,} samples")
print(f"Val   : {len(df_val):>7,} samples")
print(f"Test  : {len(df_test):>7,} samples")
print(f"Total : {len(df_train) + len(df_val) + len(df_test):>7,} samples")

In [ ]:
# Verify stratification — class proportions should be consistent across splits
def split_class_dist(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = df["class_name"].value_counts().reindex(CLASS_NAMES.values(), fill_value=0)
    pcts = (counts / counts.sum() * 100).round(1)
    return pd.DataFrame({split_name: pcts})

split_comparison = pd.concat([
    split_class_dist(df_clean, "all"),
    split_class_dist(df_train, "train"),
    split_class_dist(df_val, "val"),
    split_class_dist(df_test, "test"),
], axis=1)

print("Class proportion (%) per split:")
split_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

split_comparison.plot(kind="bar", ax=ax, colormap="Set2", edgecolor="white")
ax.set_title("Class Proportion (%) Across Splits", fontweight="bold")
ax.set_xlabel("Class")
ax.set_ylabel("Percentage")
ax.tick_params(axis="x", rotation=30)
ax.legend(title="Split", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))

plt.tight_layout()
plt.savefig("split_proportions.png", bbox_inches="tight")
plt.show()
print("Saved: split_proportions.png")

## 7. Training Metrics Review

The classifier was trained (DistilBERT, 5 epochs, CPU) and the results were saved to `ml/models/training_metrics.json`. Let's load and visualise them.

In [ ]:
METRICS_FILE = REPO_ROOT / "ml" / "models" / "training_metrics.json"

if METRICS_FILE.exists():
    with open(METRICS_FILE) as fh:
        metrics = json.load(fh)

    print(f"Model         : {metrics['model_name']}")
    print(f"Epochs        : {metrics['epochs']}")
    print(f"Best epoch    : {metrics['best_epoch']}")
    print(f"Best val acc  : {metrics['best_val_acc']:.4f}")
    print(f"Training time : {metrics['training_time_s'] / 3600:.1f} hours")
    print(f"Device        : {metrics['device']}")
    print(f"Timestamp     : {metrics['timestamp']}")
else:
    print(f"Metrics file not found at {METRICS_FILE}")
    metrics = None

In [ ]:
if metrics:
    report = metrics["final_report"]
    per_class = {
        k: v for k, v in report.items()
        if isinstance(v, dict) and "f1-score" in v
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Per-class F1
    class_labels = list(per_class.keys())
    f1_scores = [per_class[c]["f1-score"] for c in class_labels]
    bar_colors = [CLASS_COLORS.get(c, "#888") for c in class_labels]

    bars = axes[0].bar(class_labels, f1_scores, color=bar_colors, edgecolor="white")
    axes[0].set_ylim(0.9, 1.01)
    axes[0].axhline(0.95, color="red", linestyle="--", linewidth=1, label="0.95 target")
    axes[0].set_title("Per-Class F1 Score", fontweight="bold")
    axes[0].set_ylabel("F1 Score")
    axes[0].tick_params(axis="x", rotation=30)
    axes[0].legend(fontsize=9)
    for bar, score in zip(bars, f1_scores):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                     f"{score:.3f}", ha="center", va="bottom", fontsize=8)

    # Confusion matrix
    if "confusion_matrix" in report:
        cm = np.array(report["confusion_matrix"])
        cm_labels = list(CLASS_NAMES.values())
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=cm_labels,
            yticklabels=cm_labels,
            ax=axes[1],
            linewidths=0.5,
        )
        axes[1].set_title("Confusion Matrix (Val Split)", fontweight="bold")
        axes[1].set_xlabel("Predicted")
        axes[1].set_ylabel("Actual")
        axes[1].tick_params(axis="x", rotation=35)

    plt.tight_layout()
    plt.savefig("classifier_results.png", bbox_inches="tight")
    plt.show()
    print("Saved: classifier_results.png")
else:
    print("Skipping — training_metrics.json not available.")

In [ ]:
if metrics:
    report = metrics["final_report"]
    rows = []
    for c in CLASS_NAMES.values():
        if c in report:
            rows.append({
                "Class": c,
                "Precision": f"{report[c]['precision']:.4f}",
                "Recall": f"{report[c]['recall']:.4f}",
                "F1-Score": f"{report[c]['f1-score']:.4f}",
                "Support": int(report[c]["support"]),
            })
    summary_df = pd.DataFrame(rows)

    print(f"Overall accuracy : {report['accuracy']:.4f}")
    print(f"Macro avg F1     : {report['macro avg']['f1-score']:.4f}")
    print(f"Benign FPR       : {report['fpr_benign']:.4f}  (target: < 0.01)")
    print(f"ECE              : {report['ece']:.4f}  (target: < 0.05)")
    print()
    display(summary_df)

## 8. Key Takeaways

| Finding | Detail |
|---------|--------|
| **Dataset size** | ~21,600 samples across 5 classes (after cleaning) |
| **Class imbalance** | Benign 62%, direct_injection 30%; minority classes < 4% |
| **Data sources** | 7 public datasets + 2 synthetic generators |
| **Text length** | Median ~50 words; `indirect_injection` longest (~100 words) |
| **Truncation** | < 5% of samples exceed 128-token limit |
| **Stratified split** | 80/10/10 preserves class balance across train/val/test |
| **Classifier result** | 99.49% accuracy, macro F1 98.89%, benign FPR 0.44%, ECE 0.0049 |
| **All targets met** | ✅ All four primary metrics pass their thresholds |

---

### Why the class imbalance is intentional

Real healthcare AI traffic is overwhelmingly legitimate. Training on a representative distribution forces the model to generalise correctly instead of over-detecting attacks. The key constraint is keeping benign FPR < 1% — false positives delay patient care — so we explicitly monitor that metric alongside standard accuracy.

### Next steps

1. Increase `indirect_injection` diversity beyond the 15 fixed templates  
2. Source real-world indirect injection examples from adversarial NLP literature  
3. Run multi-seed evaluation (5 seeds) and report mean ± std  
4. Threshold calibration for confidence-gated routing (auto / hold / block)

---
*AEGIS — Adversarial Enforcement and Guardrail Interception System | Saint Peter's University, 2026*